In [1]:
%pip install chronos-forecasting
%pip install ipywidgets
%pip install transformers accelerate


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import findspark
import pyspark as spark
from pyspark.sql import SparkSession
from chronos import Chronos2Pipeline
from pyspark.sql.types import StructType, StructField,FloatType,TimestampType,StringType
from pyspark.sql.functions import col,to_json,struct,from_json,to_timestamp,count, collect_list
from dotenv import load_dotenv

In [2]:
load_dotenv()

False

In [3]:
chronos2_pipeline = Chronos2Pipeline.from_pretrained("../Offline-Phase/chronos2_zero_shot")

### Pandas Functions from the offline phase

In [4]:
def extract_time_features(df, timestamp_col='timestamp'):

    month = df[timestamp_col].dt.month

    df['season'] = np.select(
        [
            month.isin([12, 1, 2]),
            month.isin([3, 4, 5]),
            month.isin([6, 7, 8]),
            month.isin([9, 10, 11])
        ],
        [
            'winter',
            'spring',
            'summer',
            'autumn'
        ]
    )
    df['hour_sin'] = np.sin(2 * np.pi * df[timestamp_col].dt.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df[timestamp_col].dt.hour / 24)


    df['month_sin'] = np.sin(2 * np.pi * (df[timestamp_col].dt.month - 1) / 12)
    df['month_cos'] = np.cos(2 * np.pi * (df[timestamp_col].dt.month - 1) / 12)
    
    
    df['day_sin'] = np.sin(2 * np.pi * df[timestamp_col].dt.dayofweek / 7)
    df['day_cos'] = np.cos(2 * np.pi * df[timestamp_col].dt.dayofweek / 7)

    df['is_weekend'] = df[timestamp_col].dt.dayofweek.isin([5, 6]).astype(int)

 
    df['is_heating_season'] = df[timestamp_col].dt.month.isin([11, 12, 1, 2, 3]).astype(int)

    return df

In [5]:
def append_neighbors(df_hourly, neighbors_df, weather_cols=["humidity", "pressure","temperature", "wind_speed"], k_search=20, k_keep=5):
    # 1. Standardize the neighbor list
    # Ensure we only take the top K based on distance
    neighbors_topk = (
        neighbors_df.sort_values(["sensor_id", "distance_km"])
        .groupby("sensor_id")
        .head(k_search)
        .copy()
    )

    # Track original distance rank
    neighbors_topk['dist_rank'] = neighbors_topk.groupby("sensor_id").cumcount() + 1

    # 2. Merge with main data
    # We use 'neighbor_id' from the matrix to match 'sensorId' in the hourly data
    neighbor_values = neighbors_topk.merge(
        df_hourly[['sensorId', 'timestamp'] + weather_cols],
        left_on='neighbor_id',
        right_on='sensorId',
        how='inner'
    )

    # 3. Filter for availability
    # The 'sensor_id' here is the ORIGINAL sensor we are finding neighbors for
    available_topk = (
        neighbor_values.sort_values(['sensor_id', 'timestamp', 'dist_rank'])
        .groupby(['sensor_id', 'timestamp'])
        .head(k_keep)
        .copy()
    )

    # Create the 1, 2, 3 rank for the wide-format columns
    available_topk['final_rank'] = available_topk.groupby(['sensor_id', 'timestamp']).cumcount() + 1

    # 4. Pivot to wide format
    pivot_df = available_topk.pivot(
        index=['sensor_id', 'timestamp'],
        columns='final_rank',
        values=weather_cols
    )

    # Clean up column names: neighbor1_temp, neighbor2_temp, etc.
    if isinstance(pivot_df.columns, pd.MultiIndex):
        pivot_df.columns = [f"neighbor{rank}_{col}" for col, rank in pivot_df.columns]
    else:
        # Handle case with only one weather column
        pivot_df.columns = [f"neighbor{i}_{weather_cols[0]}" for i in pivot_df.columns]

    pivot_df = pivot_df.reset_index()

    # 5. Final Join back to original data
    df_result = df_hourly.merge(
        pivot_df,
        left_on=['sensorId', 'timestamp'],
        right_on=['sensor_id', 'timestamp'],
        how='left'
    ).drop(columns=['sensor_id'])

    return df_result

In [6]:
neighbourhood_matrix = pd.read_csv("../data/neighbors_data/skopje_neighbors.csv")
neighbourhood_matrix

,sensor_id,neighbor_id,distance_km
0,sensor_dev_60237_141,sensor_dev_10699_244,6.937128
1,sensor_dev_10699_244,sensor_dev_60237_141,6.937128
2,sensor_dev_60237_141,fefbf9e0-ff44-4b85-b968-2af046c4f4dc,3.446379
3,fefbf9e0-ff44-4b85-b968-2af046c4f4dc,sensor_dev_60237_141,3.446379
4,sensor_dev_60237_141,sensor_dev_81984_843,7.062097
...,...,...,...
34405,680b0098-7c4d-44cf-acb1-dc4031e93d34,sensor_dev_51158_685,12.425126
34406,sensor_dev_51158_685,8c5db228-0f2c-4bb5-9141-ac8b5e7f6c8a,2.571231
34407,8c5db228-0f2c-4bb5-9141-ac8b5e7f6c8a,sensor_dev_51158_685,2.571231
34408,680b0098-7c4d-44cf-acb1-dc4031e93d34,8c5db228-0f2c-4bb5-9141-ac8b5e7f6c8a,13.408616


In [7]:
def load_context():
    context_df = pd.read_csv("context_skopje.csv")
    context_df['timestamp'] = pd.to_datetime(context_df['timestamp'],utc=True)
    return context_df

In [8]:
def load_forecast():
    forecast_df = pd.read_csv('../data/raw/skopje_forecast_weather.csv')
    forecast_df['timestamp'] = pd.to_datetime(forecast_df['timestamp'],utc=True)
    forecast_df.drop(columns=['wind_direction_10m','lon','lat'],inplace=True)
    forecast_df.rename(columns={"temperature_2m":"temperature","relative_humidity_2m":"humidity","surface_pressure":"pressure","wind_speed_10m":"wind_speed"},inplace=True)
    
    forecast_df = extract_time_features(forecast_df)
    forecast_df = append_neighbors(forecast_df, neighbourhood_matrix)
    return forecast_df

In [9]:
def build_future_df(ts_df, forecast_df):

    ts_df = ts_df.copy()
    forecast_df = forecast_df.copy()

    ts_df["timestamp"] = pd.to_datetime(ts_df["timestamp"], utc=True)
    forecast_df["timestamp"] = pd.to_datetime(forecast_df["timestamp"], utc=True)

    future_rows = []

    for sensor_id in ts_df["sensorId"].unique():

        sensor_current = ts_df[ts_df["sensorId"] == sensor_id]

        base_time = sensor_current["timestamp"].max()

        future_times = [
            base_time + pd.Timedelta(hours=i)
            for i in range(1, 25)
        ]

        sensor_future = pd.DataFrame({
            "sensorId": sensor_id,
            "timestamp": future_times
        })

        sensor_forecast = forecast_df[
            forecast_df["sensorId"] == sensor_id
        ]

        sensor_future = sensor_future.merge(
            sensor_forecast,
            on=["sensorId", "timestamp"],
            how="left"
        )

        future_rows.append(sensor_future)

    future_df = pd.concat(future_rows, ignore_index=True)

    return future_df

In [10]:
def predict_target(current_df, context_df, target, pipeline, ID_COL, TIME_COL):
    context_target = context_df.copy()
    current_df_clean = current_df.copy()
    
    # 1. Strip timezones so pandas resampling and Chronos alignment work properly
    context_target[TIME_COL] = pd.to_datetime(context_target[TIME_COL], utc=True).dt.tz_convert(None)
    current_df_clean[TIME_COL] = pd.to_datetime(current_df_clean[TIME_COL], utc=True).dt.tz_convert(None)
    
    # 2. Drop any accidental duplicates from streaming micro-batch overlaps
    context_target = context_target.drop_duplicates(subset=[ID_COL, TIME_COL], keep='last')
    
    # 3. FIX FOR CHRONOS: Force strict hourly intervals and ensure min 3 rows
    fixed_dfs = []
    for sensor_id, group in context_target.groupby(ID_COL):
        # Set time as index and resample to exactly 1 hour ('h'), forward-filling any gaps
        group = group.set_index(TIME_COL).resample('h').ffill()
        
        # If a brand new sensor has less than 3 rows, Chronos will crash. 
        # We must pad it backward in time temporarily.
        if len(group) < 3:
            min_time = group.index.min()
            extra_needed = 3 - len(group)
            extra_times = [min_time - pd.Timedelta(hours=i) for i in range(extra_needed, 0, -1)]
            
            # Duplicate the oldest row to fill the required history
            pad_df = pd.DataFrame([group.iloc[0]] * extra_needed, index=extra_times)
            group = pd.concat([pad_df, group])
            
        group = group.reset_index()
        group[ID_COL] = sensor_id
        fixed_dfs.append(group)
        
    # Reassemble the clean, strictly hourly context DataFrame
    context_target = pd.concat(fixed_dfs, ignore_index=True)
    
    # Clean up the future inputs
    current_df_clean['humidity'] = current_df_clean['humidity'].astype(float)
    forecast_df = pipeline.predict_df(
        df=context_target,
        prediction_length=24,
        target=target,
        id_column=ID_COL,
        future_df=current_df_clean,
        validate_inputs=False
    )

    forecast_df = forecast_df.sort_values([ID_COL, TIME_COL])


    current_pred = (
        forecast_df
        .groupby(ID_COL)
        .first()
        .reset_index()
        .rename(columns={"predictions": target})
    )


    forecast_24h = (
        forecast_df
        .groupby(ID_COL)["predictions"]
        .apply(lambda x: x.head(24).tolist())
        .reset_index()
        .rename(columns={"predictions": f"{target}_forecast_24h"})
    )

  
    result_df = current_df.merge(
        current_pred[[ID_COL, target]],
        on=ID_COL,
        how="left"
    )


    result_df = result_df.merge(
        forecast_24h,
        on=ID_COL,
        how="left"
    )

    return result_df

In [11]:
def process_batch(current_df, context_df):
    ID_COL = "sensorId"
    TIME_COL = "timestamp"

    current_df[TIME_COL] = pd.to_datetime(current_df[TIME_COL], utc=True)
    context_df[TIME_COL] = pd.to_datetime(context_df[TIME_COL], utc=True)

    context_sorted = context_df.copy().sort_values([ID_COL, TIME_COL])

    print("Context max timestamp:", context_sorted[TIME_COL].max())

    pm10_df = predict_target(
        current_df,
        context_sorted,
        target="pm10",
        pipeline=chronos2_pipeline,
        ID_COL=ID_COL,
        TIME_COL=TIME_COL,
    )

    pm25_df = predict_target(
        current_df,
        context_sorted,
        target="pm25",
        pipeline=chronos2_pipeline,
        ID_COL=ID_COL,
        TIME_COL=TIME_COL,
    )

    return pm10_df, pm25_df

In [12]:
def write_to_kafka(df, topic):
    spark_df = spark.createDataFrame(df)

    kafka_df = spark_df.select(
        col("sensorId").cast("string").alias("key"),
        to_json(struct(*spark_df.columns)).alias("value")
    )

    kafka_df.write \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "localhost:9092") \
        .option("topic", topic) \
        .save()

In [13]:
context_df = load_context()
forecast_df = load_forecast()

def foreach_batch(batch_df, epoch_id):
    global context_df

    print(f"\nBatch received! Epoch: {epoch_id}")

    if batch_df.rdd.isEmpty():
        return

    current_df = batch_df.toPandas()

    if current_df.empty:
        print("Empty batch — skipping")
        return

    current_df = current_df.sort_values("timestamp")

    for _, row in current_df.iterrows():
        ts = row["timestamp"]

        ts_df = pd.DataFrame([r.asDict() for r in row["rows"]])

        print(f"\nProcessing timestamp: {ts}")
        print(f"Sensor count: {len(ts_df)}")

        context_max_ts = pd.to_datetime(context_df["timestamp"], utc=True).max() if not context_df.empty else None
        if context_max_ts is not None and pd.to_datetime(ts, utc=True) <= context_max_ts:
            print(f"Skipping timestamp {ts} because it is not after context max {context_max_ts}")
            continue
        
        incoming_ids = set(ts_df['sensorId'].unique())
        existing_ids = set(context_df['sensorId'].unique()) if not context_df.empty else set()
        new_ids = incoming_ids - existing_ids

        if new_ids:
            print(f"New sensors detected: {new_ids}")

            numeric_columns = [
                col for col in context_df.columns
                if col in ['temperature','wind_speed','humidity','pm10','pm25','pressure']
            ]

            city_baseline = context_df.groupby('timestamp')[numeric_columns].median().reset_index()

            proxy_rows = []

            for sid in new_ids:
                proxy_history = city_baseline.copy()
                proxy_history['sensorId'] = sid

                temp_combined = pd.concat([context_df, proxy_history], ignore_index=True)

                refined_data = append_neighbors(temp_combined, neighbourhood_matrix)
                refined_data = extract_time_features(refined_data)

                new_sensor_proxy = refined_data[refined_data['sensorId'] == sid]
                proxy_rows.append(new_sensor_proxy)

            context_df = pd.concat([context_df, *proxy_rows], ignore_index=True)

        ts_df["timestamp"] = pd.to_datetime(ts_df["timestamp"], utc=True)

        ts_df = extract_time_features(ts_df)
        ts_df = append_neighbors(ts_df, neighbourhood_matrix)

        future_df = build_future_df(ts_df,forecast_df)

        pm10_df, pm25_df = process_batch(future_df, context_df)

        if pm10_df is None or pm25_df is None:
            print("Prediction skipped")
            continue

        write_to_kafka(pm10_df, topic="FullPm10WeatherData_ZeroShot")
        write_to_kafka(pm25_df, topic="FullPm25WeatherData_ZeroShot")
        
        ts_df = ts_df.merge(
            pm10_df[['sensorId', 'timestamp', 'pm10']],
            on=['sensorId', 'timestamp'],
            how='left'
        )

        ts_df = ts_df.merge(
            pm25_df[['sensorId', 'timestamp', 'pm25']],
            on=['sensorId', 'timestamp'],
            how='left'
        )
        context_df = pd.concat([context_df, ts_df], ignore_index=True)

        context_df = (
            context_df
            .sort_values(["sensorId", "timestamp"])
            .drop_duplicates(subset=["sensorId", "timestamp"], keep="last")
            .groupby("sensorId")
            .tail(72)
            .reset_index(drop=True)
        )

        print(f"Prediction done for {ts}")
        print(f"Context size: {len(context_df)}")

# Online Phase (Main Program)

In [14]:
findspark.init()

In [15]:
spark = SparkSession.builder \
    .appName("KafkaConsumerExample") \
    .config( "spark.jars.packages","org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.7")  \
    .getOrCreate()

:: loading settings :: url = jar:file:/Users/jovan/Documents/Kodovi/RNMP/Project/venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/jovan/.ivy2/cache
The jars for the packages stored in: /Users/jovan/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a15a7b26-b7c8-4a55-afa8-bbc3e7995192;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.7 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.7 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 160ms :: artifacts dl 4ms
	:: 

In [16]:
df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribePattern", "sensor_.*") \
    .load()

In [17]:
df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [18]:
schema = StructType([
    StructField("timestamp",TimestampType(),True),
    StructField("sensorId",StringType(),True),
    StructField("lat",FloatType(),True),
    StructField("lon",FloatType(),True),
    StructField("humidity",FloatType(),True),
    StructField("pressure",FloatType(),True),
    StructField("temperature",FloatType(),True),
    StructField("wind_speed",FloatType(),True)
])

In [19]:
parsed_df = df.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), schema).alias("data")) \
    .select("data.*") \
    .drop("lat", "lon")

In [20]:
parsed_df = parsed_df.withColumn(
    "timestamp",
    to_timestamp("timestamp")
)

In [21]:
parsed_df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- humidity: float (nullable = true)
 |-- pressure: float (nullable = true)
 |-- temperature: float (nullable = true)
 |-- wind_speed: float (nullable = true)



In [22]:
grouped_df = parsed_df \
    .withWatermark("timestamp", "5 minutes") \
    .groupBy("timestamp") \
    .agg(
        collect_list(struct("*")).alias("rows"),
        count("*").alias("sensor_count")
    )

In [23]:
query = grouped_df.writeStream \
    .foreachBatch(foreach_batch) \
    .start()

query.awaitTermination()

26/08/03 22:17:28 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/18/8m8jfl3d72v4p728z2p8fn8c0000gn/T/temporary-20ede6c5-c0e3-4ffc-b33b-5854676a651a. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/08/03 22:17:28 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/08/03 22:17:28 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.



Batch received! Epoch: 0



Batch received! Epoch: 1



Batch received! Epoch: 2



Batch received! Epoch: 3


...


Batch received! Epoch: 5


26/08/03 22:18:29 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/08/03 22:18:29 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/08/03 22:18:29 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/08/03 22:18:30 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/08/03 22:18:30 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/08/03 22:18:31 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/08/03 22:18:31 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/08/03 22:18:31 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/08/03 22:18:32 WARN InternalKafkaConsumerPool: Pool exceeds its soft max size, cleaning up idle objects...
26/08/03 2


Processing timestamp: 2025-11-30 23:00:00
Sensor count: 186
Skipping timestamp 2025-11-30 23:00:00 because it is not after context max 2025-11-30 23:00:00+00:00

Processing timestamp: 2025-12-01 00:00:00
Sensor count: 186
New sensors detected: {'3c286a07-a04d-43d1-907f-e578355f242a', 'e7a05c01-1d5c-479a-a5a5-419f28cebeef', '996a42d1-c6a3-4ec1-b879-bae2a55b7777', 'a35c8770-ae2e-4639-b692-b7077e1b4e91', 'aad56180-edb5-442d-be63-b8b5f453a56d', 'a880569d-4dcc-467c-8e51-2610d272ff9c', 'sensor_dev_79777_78', 'sensor_dev_81880_63', 'sensor_dev_77125_71', '1005', 'sensor_dev_82707_917', 'sensor_dev_78021_396', 'sensor_dev_77849_684', '1f5c7035-6c1f-45ee-97f4-0a292b49710a', '0c33bee5-1139-472a-aef7-6d855bc5010d', 'ef8fbcf0-e04e-4d15-ab3c-a625a2f9245d', 'c38dabe0-8631-4f1a-b3ed-4014dacce39c', 'sensor_dev_81984_843', '32c9795d-d1e7-45e3-afd4-c302568db8c3', '7b9efe93-d604-4d6c-b2da-04c0b8882292', 'sensor_dev_82683_723', 'bb32cbe7-0391-463c-9716-d37930d1f567', 'sensor_dev_78133_229', 'sensor_dev_7

26/08/03 22:18:47 ERROR MicroBatchExecution: Query [id = 2c0b0139-01ae-45ee-81c8-0445836d0090, runId = 743c8c49-409f-420b-9c2b-0a2918748a24] terminated with error
py4j.Py4JException: An exception was raised by the Python Proxy. Return Message: Traceback (most recent call last):
  File "/Users/jovan/Documents/Kodovi/RNMP/Project/venv/lib/python3.11/site-packages/py4j/clientserver.py", line 617, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/jovan/Documents/Kodovi/RNMP/Project/venv/lib/python3.11/site-packages/pyspark/sql/utils.py", line 120, in call
    raise e
  File "/Users/jovan/Documents/Kodovi/RNMP/Project/venv/lib/python3.11/site-packages/pyspark/sql/utils.py", line 117, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
  File "/var/folders/18/8m8jfl3d72v4p728z2p8fn8c0000gn/T/ipykernel_1509/126139920.py", line 70, in foreach_batch
    pm10_df, pm25_df = p

StreamingQueryException: [STREAM_FAILED] Query [id = 2c0b0139-01ae-45ee-81c8-0445836d0090, runId = 743c8c49-409f-420b-9c2b-0a2918748a24] terminated with exception: An exception was raised by the Python Proxy. Return Message: Traceback (most recent call last):
  File "/Users/jovan/Documents/Kodovi/RNMP/Project/venv/lib/python3.11/site-packages/py4j/clientserver.py", line 617, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/jovan/Documents/Kodovi/RNMP/Project/venv/lib/python3.11/site-packages/pyspark/sql/utils.py", line 120, in call
    raise e
  File "/Users/jovan/Documents/Kodovi/RNMP/Project/venv/lib/python3.11/site-packages/pyspark/sql/utils.py", line 117, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
  File "/var/folders/18/8m8jfl3d72v4p728z2p8fn8c0000gn/T/ipykernel_1509/126139920.py", line 70, in foreach_batch
    pm10_df, pm25_df = process_batch(future